In [68]:
import pandas as pd
import re

pd.set_option('display.max_colwidth', None)

INPUT_PATH = "./ohchr_instruments_detailed.csv"
OUTPUTPATH = "./ohchr_instruments_detailed-instit.csv"

In [69]:
# Read dataframe
df = pd.read_csv(INPUT_PATH)

# --------------------------------------------------
# Helper functions
# --------------------------------------------------

def extract_info(text):
    """
    Extract the line immediately following 'BY'.
    """
    if pd.isna(text):
        return None

    match = re.search(
        r"(?im)^BY\s*\n([^\n\r]+)",
        str(text)
    )

    return match.group(1).strip() if match else None


def extract_adoption_date(text):
    """
    Extract the line immediately following 'ADOPTED'.
    """
    if pd.isna(text):
        return None

    match = re.search(
        r"(?im)^ADOPTED\s*\n([^\n\r]+)",
        str(text)
    )

    return match.group(1).strip() if match else None


# --------------------------------------------------
# Create new columns
# --------------------------------------------------

df["info"] = df["content"].apply(extract_info)
# df["adoption_date"] = df["content"].apply(extract_adoption_date) # Not necessary because the 'adoption_by' already captures this
df.rename(columns={"adopted_by": "adoption_date"}, inplace=True)
df["institution"] = None
df["resolution"] = None
df["event"] = None
df["relation"] = None
df["location"] = None

# --------------------------------------------------
# Corrections
# --------------------------------------------------
url = 'https://www.ohchr.org/en/instruments-mechanisms/instruments/standard-rules-equalization-opportunities-persons-disabilities'
df.loc[df["url"] == url, "institution"] = "General Assembly"
df.loc[df["url"] == url, "resolution"] = "A/RES/48/96"
df.loc[df["url"] == url, "event"] = "Forty-eighth session" # Easy to infere becasue of A/RES/48 (session N. 48)

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/slavery-convention"
df.loc[df["url"] == url, "institution"] = "League of Nations"
df.loc[df["url"] == url, "resolution"] = "LoN-1414"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/statute-international-tribunal-prosecution-persons-responsible"
df.loc[df["url"] == url, "institution"] = "Security Council"
df.loc[df["url"] == url, "resolution"] = "S/RES/827(1993)"
df.loc[df["url"] == url, "event"] = "3217th meeting"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/abolition-forced-labour-convention-1957-no-105"
df.loc[df["url"] == url, "institution"] = "International Labour Organisation"
df.loc[df["url"] == url, "event"] = "Fortieth session"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/convention-against-discrimination-education"
df.loc[df["url"] == url, "institution"] = "United Nations Educational, Scientific and Cultural Organization"
df.loc[df["url"] == url, "event"] = "General Conference"  # no session number given in info

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/declaration-race-and-racial-prejudice"
df.loc[df["url"] == url, "institution"] = "United Nations Educational, Scientific and Cultural Organization"
df.loc[df["url"] == url, "event"] = "Twentieth session"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/discrimination-employment-and-occupation-convention-1958-no-111"
df.loc[df["url"] == url, "institution"] = "International Labour Organisation"
df.loc[df["url"] == url, "event"] = "Forty-second session"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/employment-policy-convention-1964-no-122"
df.loc[df["url"] == url, "institution"] = "International Labour Organisation"
df.loc[df["url"] == url, "event"] = "Forty-eighth session"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/equal-remuneration-convention-1951-no-100"
df.loc[df["url"] == url, "institution"] = "International Labour Organisation"
df.loc[df["url"] == url, "event"] = "Thirty-fourth session"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/basic-principles-independence-judiciary"
df.loc[df["url"] == url, "event"] = "Seventh United Nations Congress on the Prevention of Crime and the Treatment of Offenders, Milan, 26 August-6 September 1985"
df.loc[df["url"] == url, "location"] = "Milan"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/basic-principles-role-lawyers"
df.loc[df["url"] == url, "event"] = "Eighth United Nations Congress on the Prevention of Crime and the Treatment of Offenders, Havana, Cuba"
df.loc[df["url"] == url, "location"] = "Havana, Cuba"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/basic-principles-use-force-and-firearms-law-enforcement"
df.loc[df["url"] == url, "event"] = "Eighth United Nations Congress on the Prevention of Crime and the Treatment of Offenders, Havana, Cuba, 27 August-7 September 1990"
df.loc[df["url"] == url, "location"] = "Havana, Cuba"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/convention-relating-status-stateless-persons"
idx = df.index[df["url"] == url][0]
df.loc[df["url"] == url, "event"] = "Conference of Plenipotentiaries"
df.at[idx, "relation"] = [{
    "A_entity": "E/RES/526A(XVII)",     # ECOSOC resolution 526 A (XVII), 26 April 1954
    "relation_type": "convened by",
    "B_entity": idx
}]


to_correct = df[
    df["institution"].isna() &
    df["event"].isna() &
    df["location"].isna()
]


In [ ]:
import json

# ---------------------------------------------------------
# Pattern D: real relation (convened_by) -> goes in 'relation'
# ---------------------------------------------------------


# ---------------------------------------------------------
# Pattern C: ILO conventions -> known organ + session, no relation
# ---------------------------------------------------------
forced_labour = "https://www.ohchr.org/en/instruments-mechanisms/instruments/forced-labour-convention-1930-no-29"
df.loc[df["url"] == forced_labour, "institution"] = "International Labour Organisation"
df.loc[df["url"] == forced_labour, "event"] = "Fourteenth session"

freedom_association = "https://www.ohchr.org/en/instruments-mechanisms/instruments/freedom-association-and-protection-right-organize-convention"
df.loc[df["url"] == freedom_association, "institution"] = "International Labour Organisation"
df.loc[df["url"] == freedom_association, "event"] = "Thirty-first session"

indigenous_tribal = "https://www.ohchr.org/en/instruments-mechanisms/instruments/indigenous-and-tribal-peoples-convention-1989-no-169"
df.loc[df["url"] == indigenous_tribal, "institution"] = "International Labour Organisation"
df.loc[df["url"] == indigenous_tribal, "event"] = "Seventy-sixth session"

minimum_age = "https://www.ohchr.org/en/instruments-mechanisms/instruments/minimum-age-convention-1973-no-138"
df.loc[df["url"] == minimum_age, "institution"] = "International Labour Organisation"
df.loc[df["url"] == minimum_age, "event"] = "Fifty-eighth session"

# ---------------------------------------------------------
# Pattern B: event only, institution unknown, no relation
# ---------------------------------------------------------
geneva_civilians = "https://www.ohchr.org/en/instruments-mechanisms/instruments/geneva-convention-relative-protection-civilian-persons-time-war"
df.loc[df["url"] == geneva_civilians, "event"] = (
    "Diplomatic Conference for the Establishment of International Conventions "
    "for the Protection of Victims of War (Geneva, 21 April - 12 August 1949)"
)

geneva_pow = "https://www.ohchr.org/en/instruments-mechanisms/instruments/geneva-convention-relative-treatment-prisoners-war"
df.loc[df["url"] == geneva_pow, "event"] = (
    "Diplomatic Conference for the Establishment of International Conventions "
    "for the Protection of Victims of War (Geneva, 21 April - 12 August 1949)"
)

guidelines_prosecutors = "https://www.ohchr.org/en/instruments-mechanisms/instruments/guidelines-role-prosecutors"
df.loc[df["url"] == guidelines_prosecutors, "event"] = (
    "Eighth United Nations Congress on the Prevention of Crime and the Treatment of Offenders"
)

# ---------------------------------------------------------
# Pattern A: direct organ + resolution, no session, no relation
# ---------------------------------------------------------
guidelines_children = "https://www.ohchr.org/en/instruments-mechanisms/instruments/guidelines-action-children-criminal-justice-system"
df.loc[df["url"] == guidelines_children, "institution"] = "Economic and Social Council"
df.loc[df["url"] == guidelines_children, "resolution"] = "E/RES/1997/30"

principles_extralegal = "https://www.ohchr.org/en/instruments-mechanisms/instruments/principles-effective-prevention-and-investigation-extra-legal"
df.loc[df["url"] == principles_extralegal, "institution"] = "Economic and Social Council"
df.loc[df["url"] == principles_extralegal, "resolution"] = "E/RES/1989/65"

In [70]:
print(df.columns)

Index(['title', 'url', 'adoption_date', 'content', 'pdf_url', 'info',
       'institution', 'resolution', 'event', 'relation', 'location'],
      dtype='str')


In [71]:
result = to_correct[~to_correct["info"].str.contains("General Assembly", case=False, na=False)]
print(result[["url", "info"]].head(10))

                                                                                                                            url  \
23                     https://www.ohchr.org/en/instruments-mechanisms/instruments/convention-relating-status-stateless-persons   
45                              https://www.ohchr.org/en/instruments-mechanisms/instruments/forced-labour-convention-1930-no-29   
46     https://www.ohchr.org/en/instruments-mechanisms/instruments/freedom-association-and-protection-right-organize-convention   
48  https://www.ohchr.org/en/instruments-mechanisms/instruments/geneva-convention-relative-protection-civilian-persons-time-war   
49               https://www.ohchr.org/en/instruments-mechanisms/instruments/geneva-convention-relative-treatment-prisoners-war   
50               https://www.ohchr.org/en/instruments-mechanisms/instruments/guidelines-action-children-criminal-justice-system   
51                                      https://www.ohchr.org/en/instruments-mechan

In [72]:
import numpy as np


# ------------------------------------------------------------------
# Pattern C: explicit "convened by X resolution Y" -> relation
# institution unknown at the instrument level (the direct adopting body
# is the Conference itself), the convening act is captured in 'relation'
# ------------------------------------------------------------------

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/convention-relating-status-stateless-persons"
idx = df.index[df["url"] == url][0]
df.loc[df["url"] == url, "event"] = "Conference of Plenipotentiaries"
df.at[idx, "relation"] = [{
    "A_entity": "E/RES/526A(XVII)",   # ECOSOC resolution that performed the act
    "relation_type": "convened by",
    "B_entity": idx
    }]


In [73]:
# known events
keys = ["convened under", "endorsed by", "pursuance of"]

mask_general_assembly = ~df["institution"].str.contains("General Assembly", case=False, na=False)

mask_keys = df["institution"].str.contains("resolution", case=False, na=False)

result = df[mask_general_assembly & mask_keys]

print(result[["url", "institution"]])


Empty DataFrame
Columns: [url, institution]
Index: []


In [74]:
# known events
keys = ["convened under", "endorsed by", "pursuance of"]

mask_general_assembly = df["institution"].str.contains("General Assembly", case=False, na=False)

mask_keys = df["institution"].str.contains("|".join(keys), case=False, na=False)

result = df.loc[mask_general_assembly & mask_keys, "institution"].to_list()

result


[]

In [75]:
keys = ["convened under", "endorsed by", "pursuance of"]

mask_general_assembly = df["institution"].str.contains("General Assembly", case=False, na=False)

mask_not_keys = ~df["institution"].str.contains("|".join(keys), case=False, na=False)

result = df.loc[mask_general_assembly & mask_not_keys, "institution"].to_list()

result

['General Assembly']